Cell 1 — Install Dependencies

In [ ]:
!pip install -q groq pandas reportlab

Cell 2 — Load Groq API Key from Colab Secrets

In [ ]:
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

print("Groq key loaded successfully!")

Groq key loaded successfully!


Cell 3 — Initialize Groq Client

In [ ]:
from groq import Groq

client = Groq(
    api_key=GROQ_API_KEY
)

print("Groq client initialized")

Groq client initialized


Cell 4 — Upload Transcript File

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving sample_transcript.txt to sample_transcript.txt


Cell 5 — Read Transcript

In [ ]:
filename = list(uploaded.keys())[0]

with open(filename, "r", encoding="utf-8") as f:
    transcript = f.read()

print("Transcript Length:", len(transcript))
print("\nPreview:\n")
print(transcript[:1000])

Transcript Length: 494

Preview:

John: Good morning everyone. Today we'll review the website redesign project.

Sarah: The UI mockups are complete and ready for review.

John: Great. Let's finalize the design by Friday.

Mike: I will update the backend API integration by next Wednesday.

Sarah: I'll prepare the client presentation before Monday.

John: Decision made: launch date remains July 15.

Mike: We should also improve dashboard performance.

John: Good idea. Let's investigate optimization opportunities next sprint.


Cell 6 — Create Prompt

In [ ]:
prompt = f"""
You are an AI Meeting Assistant.

Analyze the meeting transcript below.

Return ONLY valid JSON.

Required format:

{{
  "meeting_type": "",
  "summary": [
    "",
    "",
    "",
    "",
    ""
  ],
  "decisions": [
    ""
  ],
  "action_items": [
    {{
      "task": "",
      "assignee": "",
      "deadline": ""
    }}
  ]
}}

Meeting Types:
- Standup
- Planning
- Review
- Brainstorm
- Other

Transcript:

{transcript}
"""

Cell 7 — Send Transcript to Groq

In [ ]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.1
)

result = response.choices[0].message.content

print(result)

```
{
  "meeting_type": "Review",
  "summary": [
    "The team reviewed the website redesign project.",
    "The UI mockups are complete and ready for review.",
    "The design needs to be finalized by Friday.",
    "The backend API integration will be updated by next Wednesday.",
    "The client presentation will be prepared before Monday."
  ],
  "decisions": [
    "The launch date remains July 15."
  ],
  "action_items": [
    {
      "task": "Finalize the design",
      "assignee": "Team",
      "deadline": "Friday"
    },
    {
      "task": "Update backend API integration",
      "assignee": "Mike",
      "deadline": "Next Wednesday"
    },
    {
      "task": "Prepare client presentation",
      "assignee": "Sarah",
      "deadline": "Monday"
    },
    {
      "task": "Investigate dashboard optimization opportunities",
      "assignee": "Team",
      "deadline": "Next sprint"
    }
  ]
}
```


Cell 8 — Parse JSON Output

In [ ]:
import json

# Remove markdown code fences
clean_result = result.replace("```json", "").replace("```", "").strip()

data = json.loads(clean_result)

print("JSON Parsed Successfully")

JSON Parsed Successfully


Cell 9 — Display Meeting Summary

In [ ]:
print("="*60)
print("MEETING TYPE")
print("="*60)

print(data["meeting_type"])

print("\n")

print("="*60)
print("SUMMARY")
print("="*60)

for point in data["summary"]:
    print("•", point)

MEETING TYPE
Review


SUMMARY
• The team reviewed the website redesign project.
• The UI mockups are complete and ready for review.
• The design needs to be finalized by Friday.
• The backend API integration will be updated by next Wednesday.
• The client presentation will be prepared before Monday.


Cell 10 — Display Decisions

In [ ]:
print("="*60)
print("DECISIONS MADE")
print("="*60)

for decision in data["decisions"]:
    print("•", decision)

DECISIONS MADE
• The launch date remains July 15.


Cell 11 — Display Action Items Table

In [ ]:
import pandas as pd

action_df = pd.DataFrame(data["action_items"])

action_df

,task,assignee,deadline
0,Finalize the design,Team,Friday
1,Update backend API integration,Mike,Next Wednesday
2,Prepare client presentation,Sarah,Monday
3,Investigate dashboard optimization opportunities,Team,Next sprint


Cell 12 — Generate Email Summary

In [ ]:
def generate_email(data):

    summary = "\n".join(
        [f"• {item}" for item in data["summary"]]
    )

    decisions = "\n".join(
        [f"• {item}" for item in data["decisions"]]
    )

    actions = "\n".join(
        [
            f"• {item['assignee']} → {item['task']} (Deadline: {item['deadline']})"
            for item in data["action_items"]
        ]
    )

    email = f"""
MEETING SUMMARY REPORT

Meeting Type:
{data['meeting_type']}

--------------------------------------------------

SUMMARY

{summary}

--------------------------------------------------

DECISIONS

{decisions}

--------------------------------------------------

ACTION ITEMS

{actions}

--------------------------------------------------

Generated by AI Meeting Assistant
"""

    return email

email_content = generate_email(data)

print(email_content)


MEETING SUMMARY REPORT

Meeting Type:
Review

--------------------------------------------------

SUMMARY

• The team reviewed the website redesign project.
• The UI mockups are complete and ready for review.
• The design needs to be finalized by Friday.
• The backend API integration will be updated by next Wednesday.
• The client presentation will be prepared before Monday.

--------------------------------------------------

DECISIONS

• The launch date remains July 15.

--------------------------------------------------

ACTION ITEMS

• Team → Finalize the design (Deadline: Friday)
• Mike → Update backend API integration (Deadline: Next Wednesday)
• Sarah → Prepare client presentation (Deadline: Monday)
• Team → Investigate dashboard optimization opportunities (Deadline: Next sprint)

--------------------------------------------------

Generated by AI Meeting Assistant



Cell 13 — Save Email as TXT File

In [ ]:
with open("meeting_summary.txt", "w", encoding="utf-8") as f:
    f.write(email_content)

print("meeting_summary.txt created")

meeting_summary.txt created


Cell 14 — Generate PDF Report

In [ ]:
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer
)

from reportlab.lib.styles import getSampleStyleSheet

pdf = SimpleDocTemplate("Meeting_Report.pdf")

styles = getSampleStyleSheet()

elements = []

elements.append(
    Paragraph("Meeting Summary Report",
              styles['Title'])
)

elements.append(Spacer(1,12))

elements.append(
    Paragraph(
        f"Meeting Type: {data['meeting_type']}",
        styles['Heading2']
    )
)

elements.append(Spacer(1,12))

elements.append(
    Paragraph("Summary", styles['Heading2'])
)

for item in data["summary"]:
    elements.append(
        Paragraph(f"• {item}", styles['BodyText'])
    )

elements.append(Spacer(1,12))

elements.append(
    Paragraph("Decisions", styles['Heading2'])
)

for item in data["decisions"]:
    elements.append(
        Paragraph(f"• {item}", styles['BodyText'])
    )

elements.append(Spacer(1,12))

elements.append(
    Paragraph("Action Items", styles['Heading2'])
)

for item in data["action_items"]:
    elements.append(
        Paragraph(
            f"• {item['assignee']} → "
            f"{item['task']} "
            f"(Deadline: {item['deadline']})",
            styles['BodyText']
        )
    )

pdf.build(elements)

print("Meeting_Report.pdf created")

Meeting_Report.pdf created


Cell 15 — Download Generated Files

In [ ]:
from google.colab import files

files.download("Meeting_Report.pdf")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>